In [7]:
!pip -q install streamlit pyngrok scikit-learn pandas numpy joblib

In [8]:
from google.colab import files
uploaded = files.upload()

Saving realty_data.csv to realty_data.csv


In [9]:
%%writefile train_model.py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
import joblib, json
from pathlib import Path

RANDOM_STATE = 42
OUT_DIR = Path("models")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv("realty_data.csv")
print("Размер данных:", df.shape)
print(df.head())

target = "price"
features = [col for col in df.columns if col != target]

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", LinearRegression())
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)
print(f"MAE: {mae:.2f}")
print(f"R2: {r2:.3f}")

joblib.dump(model, OUT_DIR / "realty_model.pkl")
metadata = {
    "features": features,
    "defaults": X.median(numeric_only=True).to_dict()
}
with open(OUT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Модель сохранена.")


Overwriting train_model.py


In [10]:
!python train_model.py

Размер данных: (98822, 17)
          product_name  ...               source
0  3-комнатная, 137 м²  ...                 ЦИАН
1      Студия, 16,7 м²  ...              Домклик
2   3-комнатная, 76 м²  ...  Яндекс.Недвижимость
3   1-комнатная, 24 м²  ...          Новострой-М
4  3-комнатная, 126 м²  ...              Домклик

[5 rows x 17 columns]
Traceback (most recent call last):
  File "/content/train_model.py", line 31, in <module>
    model.fit(X_train, y_train)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py", line 654, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py", line 588, in _fit
    X, fitted_transformer = fit_transform_one_cached(

In [23]:
%%writefile streamlit_app.py
import json
from pathlib import Path
import joblib
import pandas as pd
import streamlit as st

# ==== пути к артефактам ====
MODEL_DIR = Path("models")
MODEL_PATH = MODEL_DIR / "house_price_model.pkl"   # ← у тебя так называется файл
META_PATH  = MODEL_DIR / "metadata.json"

st.set_page_config(page_title="Прогноз стоимости недвижимости", page_icon="🏠")
st.title("🏠 Прогноз стоимости недвижимости")

@st.cache_resource
def load_artifacts():
    model = joblib.load(MODEL_PATH)
    with open(META_PATH, "r", encoding="utf-8") as f:
        meta = json.load(f)
    # поддерживаем оба формата метаданных:
    features = meta.get("features") or meta.get("feature_names")
    defaults = meta.get("defaults", {})
    if not features:
        raise ValueError("В metadata.json нет ключей 'features' или 'feature_names'.")
    return model, features, defaults, meta

model, feature_names, defaults, meta = load_artifacts()

with st.expander("ℹ️ О модели", expanded=False):
    st.write(meta)

st.subheader("Введите значения признаков")
user = {}
for f in feature_names:
    # безопасное значение по умолчанию
    val = float(defaults.get(f, 0.0)) if isinstance(defaults, dict) else 0.0
    user[f] = st.number_input(f, value=val)

if st.button("🔮 Предсказать цену"):
    X = pd.DataFrame([user], columns=feature_names).astype(float)
    y = float(model.predict(X)[0])
    st.success(f"💰 Прогнозируемая стоимость: **{y:,.2f}**")
    with st.expander("Введённые значения", expanded=False):
        st.json(user)

Overwriting streamlit_app.py


In [24]:
import subprocess, time, re, os, signal, sys

port = 8501
st_proc = subprocess.Popen(
    ["streamlit", "run", "streamlit_app.py", "--server.port", str(port), "--server.headless", "true"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

cf_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{port}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

print("⏳ Поднимаем тоннель...")

public_url = None
deadline = time.time() + 30
while time.time() < deadline:
    line = cf_proc.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    m = re.search(r"https?://[0-9a-zA-Z\-\.]+trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

if public_url:
    print("🌐 Откройте приложение:", public_url)
else:
    print("❌ Не удалось получить публичный URL из cloudflared логов.")
    for _ in range(30):
        line = cf_proc.stdout.readline()
        if not line:
            break
        print(line.rstrip())

time.sleep(3)
for _ in range(10):
    line = st_proc.stdout.readline()
    if not line:
        break
    print(line.rstrip())


cloudflared: Text file busy
⏳ Поднимаем тоннель...
🌐 Откройте приложение: https://thousands-worth-rolls-fairfield.trycloudflare.com


2025-10-25 11:50:46.687 Port 8501 is already in use
